# Modelling: first model to predict the age of an abalone

We predict the number of rings (`Rings`, age = Rings + 1.5 years) from the physical measurements,
following the conclusions of `eda.ipynb`:
- remove the few physically inconsistent rows,
- one-hot encode `Sex`,
- start with a linear regression baseline, then compare with a non-linear model.

Every experiment is tracked with **MLflow** (parameters, metrics and model).
The tracking data is stored locally in `notebooks/mlflow.db` and `notebooks/mlruns/`
and is not versioned. To compare the runs in the MLflow UI, run from the `notebooks/` folder:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --workers 1
```

# Imports

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Configuration

In [2]:
DATA_PATH = "../data/abalone.csv"
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENT_NAME = "abalone-age-prediction"

TARGET = "Rings"
CATEGORICAL_FEATURES = ["Sex"]
NUMERICAL_FEATURES = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
]
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Data preparation

We remove the inconsistent rows identified in the EDA (zero or extreme height,
shucked weight greater than whole weight).

In [3]:
def load_data(path: str) -> pd.DataFrame:
    """Load the abalone dataset from a CSV file."""
    return pd.read_csv(path)


def remove_inconsistent_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Remove physically impossible rows (zero/extreme height, shucked weight > whole weight)."""
    is_valid = (
        (df["Height"] > 0)
        & (df["Height"] <= 0.4)
        & (df["Shucked weight"] <= df["Whole weight"])
    )
    return df[is_valid].reset_index(drop=True)


df = remove_inconsistent_rows(load_data(DATA_PATH))
print(f"Dataset shape after cleaning: {df.shape}")

Dataset shape after cleaning: (4169, 9)


In [4]:
X = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

Train: 3335 rows | Test: 834 rows


# Modelling

Each model is wrapped in a scikit-learn `Pipeline` with the same preprocessing
(one-hot encoding of `Sex`, standardization of the numerical features), so that the
fitted pipeline can later be reused as is for inference.

In [5]:
def build_pipeline(model) -> Pipeline:
    """Build a pipeline chaining the preprocessing steps and the given regressor."""
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES,
            ),
            ("numerical", StandardScaler(), NUMERICAL_FEATURES),
        ]
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])


def evaluate(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    """Compute the regression metrics used to compare the models."""
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [6]:
# sklearn types MLflow needs to trust to serialize the tree-based models (skops format)
SKOPS_TRUSTED_TYPES = ["sklearn.tree._tree.Tree"]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


def train_and_log(run_name: str, model) -> dict[str, float]:
    """Train a pipeline with the given model, evaluate it on the test set and log the run to MLflow."""
    pipeline = build_pipeline(model)

    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, y_train)
        metrics = evaluate(y_test, pipeline.predict(X_test))

        mlflow.log_param("model_type", type(model).__name__)
        mlflow.log_params(model.get_params())
        mlflow.log_params(
            {"split_test_size": TEST_SIZE, "split_random_state": RANDOM_STATE}
        )
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            input_example=X_train.head(),
            skops_trusted_types=SKOPS_TRUSTED_TYPES,
        )

    print(f"{run_name}: {metrics}")
    return metrics

2026/10/22 11:51:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/22 11:51:29 INFO mlflow.store.db.utils: Updating database tables


2026/10/22 11:51:32 INFO mlflow.tracking.fluent: Experiment with name 'abalone-age-prediction' does not exist. Creating a new experiment.


## Baseline: linear regression

In [7]:
linear_regression_metrics = train_and_log("linear_regression", LinearRegression())

2026/10/22 11:51:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


linear_regression: {'rmse': 2.1104868015460516, 'mae': 1.51709338023255, 'r2': 0.6162811786551039}


## Regularized linear model: Ridge

In [8]:
ridge_metrics = train_and_log("ridge", Ridge(alpha=1.0))

2026/10/22 11:52:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ridge: {'rmse': 2.111885112387569, 'mae': 1.517890318808911, 'r2': 0.6157725415606722}


## Non-linear model: random forest

The EDA showed a non-linear relationship between the measurements and the number of rings,
which a tree-based model can capture.

In [9]:
random_forest_metrics = train_and_log(
    "random_forest",
    RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1
    ),
)

2026/10/22 11:52:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


random_forest: {'rmse': 2.1307547830854556, 'mae': 1.4920710060470619, 'r2': 0.6088757307405253}


# Comparison of the experiments

In [10]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME], order_by=["metrics.rmse ASC"]
)
runs[["tags.mlflow.runName", "metrics.rmse", "metrics.mae", "metrics.r2", "start_time"]]

,tags.mlflow.runName,metrics.rmse,metrics.mae,metrics.r2,start_time
0,linear_regression,2.110487,1.517093,0.616281,2026-10-22 09:51:33.227000+00:00
1,ridge,2.111885,1.517890,0.615773,2026-10-22 09:51:52.037000+00:00
2,random_forest,2.130755,1.492071,0.608876,2026-10-22 09:52:09.593000+00:00


# Conclusions

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear regression | 2.11 | 1.52 | 0.62 |
| Ridge (alpha=1) | 2.11 | 1.52 | 0.62 |
| Random forest | 2.13 | 1.49 | 0.61 |

- The three models reach very similar performance: the predictions are off by about **1.5 rings
  on average** (MAE) and the measurements explain around **60% of the variance** of the age (R²).
- The **linear regression** has the best RMSE. Regularization (Ridge) brings nothing, and the random
  forest is only marginally better on MAE despite the non-linearity observed in the EDA: a large part
  of the error seems to come from the natural variability of the age for a given size, especially for
  old abalones, rather than from the choice of model.
- We keep the **linear regression** as our first model: it is the simplest, fastest and most
  interpretable, which makes it a good candidate to industrialize in the next steps (training and
  inference modules, Prefect workflows, API).
- Possible improvements: tune the models with cross-validation (the results above rely on a single
  train/test split), transform the target (e.g. log of the number of rings) or engineer features
  such as weight ratios.